# 🐭 Q-Learning: 网格世界里的老鼠找奶酪

这个 notebook 实现一个完整的 Q-learning 智能体——一只老鼠在 4×4 的网格里学找奶酪。

**纯 NumPy 实现，不带任何神经网络和 RL 框架。** 你可以看到算法的每一行。

**前提**：你已经看过 [01_hello_rl.ipynb](01_hello_rl.ipynb) 了。

## 1. 搭建网格世界

一个 4×4 的方格：
- 🐭 老鼠从左上角 (0,0) 出发
- 🧀 奶酪在右下角 (3,3)
- ⚡ 陷阱在 (1,1) 和 (2,2)

到达奶酪 → 奖励 +10
掉到陷阱 → 惩罚 -10
普通格子 → 不奖不罚 (0)

In [1]:
import numpy as np

# 4×4 的网格世界
GRID_SIZE = 4

# 奖励地图：奶酪=+10, 陷阱=-10, 空地=0
reward_map = np.zeros((GRID_SIZE, GRID_SIZE))
reward_map[3, 3] = 10   # 🧀 奶酪
reward_map[1, 1] = -10  # ⚡ 陷阱
reward_map[2, 2] = -10  # ⚡ 陷阱

print("奖励地图 (top-left 是 0,0):")
print(reward_map)

# 动作：上(0) 下(1) 左(2) 右(3)
ACTIONS = ["↑上", "↓下", "←左", "→右"]
ACTION_DELTA = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # (行变化, 列变化)

奖励地图 (top-left 是 0,0):
[[  0.   0.   0.   0.]
 [  0. -10.   0.   0.]
 [  0.   0. -10.   0.]
 [  0.   0.   0.  10.]]


In [2]:
def take_action(state, action):
    """在状态 state 执行动作 action，返回 (新状态, 奖励, 是否结束)"""
    row, col = state
    drow, dcol = ACTION_DELTA[action]
    new_row = max(0, min(GRID_SIZE - 1, row + drow))
    new_col = max(0, min(GRID_SIZE - 1, col + dcol))
    
    reward = reward_map[new_row, new_col]
    done = (reward != 0)  # 到达特殊格子就结束
    return (new_row, new_col), reward, done

# 测试：从 (0,0) 往下走
print("从 (0,0) 往下走:", take_action((0, 0), 1))
print("从 (0,0) 往右走:", take_action((0, 0), 3))

从 (0,0) 往下走: ((1, 0), 0.0, False)
从 (0,0) 往右走: ((0, 1), 0.0, False)


## 2. Q 表：给每个"状态×动作"打分

老鼠在 4×4=16 个格子里，每个格子有 4 种走法。
Q 表就是一张 16×4 的表格：`Q[位置][动作] = 在这个位置做这个动作大概能得多少分`。

In [3]:
# TODO: 初始化 Q 表（16 个格子 × 4 个动作，全填 0）
# 提示: 用 numpy 的 zeros 函数
# Q = ???

# 写完了取消下面这行的注释来验证：
# assert Q.shape == (16, 4), f"形状不对，应该是 (16,4)，当前是 {Q.shape}"

In [4]:
# 答案（如果上面写完了可以跳过不看）
Q = np.zeros((16, 4))
print(f"Q 表形状: {Q.shape}")
print(f"Q[0] (在格子0时的四个动作得分): {Q[0]}")

Q 表形状: (16, 4)
Q[0] (在格子0时的四个动作得分): [0. 0. 0. 0.]


## 3. Q-Learning 算法

每一步的更新规则：

```
Q[旧状态][旧动作] += 学习率 × (奖励 + 折扣因子 × max(Q[新状态]) - Q[旧状态][旧动作])
                   ↑                    ↑                              ↑
              这次学多少        这次的回报+未来最好的回报            旧记录
```

### 公式拆解（逐个解释每个部分）：

- `奖励`：这一步实际得到的分数（踩到奶酪 +10，掉陷阱 -10）
- `折扣因子 × max(Q[新状态])`：**预估的未来回报**（到了新位置后，最好的那种走法能拿多少分）
- `奖励 + 折扣 × 未来`：这一步的"总分"（眼前的奖励 + 未来的奖励）
- 减去 `Q[旧状态][旧动作]`：比原来预估的好多少？
- `学习率 ×`：只改一点点

**关键洞察**：Q 表不仅记录了"这一步得多少分"，还**把未来能得的分也"倒灌"回来**——奶酪的 +10 会在多轮训练后逐渐传播到周围的格子。

In [5]:
# 训练参数
EPISODES = 500          # 训练回合数
LEARNING_RATE = 0.1     # 学习率
GAMMA = 0.9             # 折扣因子（越接近 1 越看重未来）
EPSILON = 0.1           # 探索率（10% 随机，90% 选最佳）

Q = np.zeros((16, 4))   # 重置 Q 表
np.random.seed(42)

for episode in range(EPISODES):
    state = (0, 0)  # 每轮从左上角出发
    done = False
    
    while not done:
        state_idx = state[0] * GRID_SIZE + state[1]  # (row,col) → 0~15
        
        # ε-greedy: 90% 选最佳, 10% 随机（探索！）
        if np.random.random() < EPSILON:
            action = np.random.randint(4)
        else:
            action = np.argmax(Q[state_idx])
        
        # 执行动作
        next_state, reward, done = take_action(state, action)
        next_idx = next_state[0] * GRID_SIZE + next_state[1]
        
        # Q-Learning 更新！这就是核心！
        Q[state_idx, action] += LEARNING_RATE * (
            reward + GAMMA * np.max(Q[next_idx]) - Q[state_idx, action]
        )
        
        state = next_state

print(f"训练完成！{EPISODES} 回合")
print(f"\n终点的得分应该最高（奶酪+10的奖励会向前扩散）")
print(f"陷阱附近的得分应该为负")

训练完成！500 回合

终点的得分应该最高（奶酪+10的奖励会向前扩散）
陷阱附近的得分应该为负


In [6]:
# 可视化 Q 表：每个格子的"价值" = max(Q[这个格子])
value_grid = np.max(Q.reshape(4, 4, 4), axis=2)

print("每个格子的最大 Q 值（值越高 = 老鼠越想来）：")
for row in range(GRID_SIZE):
    line = ""
    for col in range(GRID_SIZE):
        if (row, col) == (3, 3):
            line += "🧀    "
        elif (row, col) in [(1, 1), (2, 2)]:
            line += "⚡    "
        else:
            line += f"{value_grid[row, col]:5.1f} "
    print(line)

print(f"\n老鼠在 (0,0) 的最佳动作: {ACTIONS[np.argmax(Q[0])]}")
print(f"老鼠在 (0,1) 的最佳动作: {ACTIONS[np.argmax(Q[1])]}")
print(f"老鼠在 (0,3) 的最佳动作: {ACTIONS[np.argmax(Q[3])]}")

每个格子的最大 Q 值（值越高 = 老鼠越想来）：
  0.0   0.0   0.0   0.0 
  0.0 ⚡      0.0   0.0 
  0.0   0.0 ⚡      0.0 
  0.0   0.0   0.0 🧀    

老鼠在 (0,0) 的最佳动作: ↑上
老鼠在 (0,1) 的最佳动作: ↑上
老鼠在 (0,3) 的最佳动作: ↑上


In [27]:
# 让训练好的老鼠走一遍，看看路径
state = (0, 0)
path = [state]
done = False

while not done and len(path) < 20:
    state_idx = state[0] * GRID_SIZE + state[1]
    action = np.argmax(Q[state_idx])  # 不做探索了，纯利用
    state, reward, done = take_action(state, action)
    path.append(state)

# 画出网格和路径
grid = [["·" for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]
grid[3][3] = "🧀"
grid[1][1] = "⚡"
grid[2][2] = "⚡"

for i, (r, c) in enumerate(path):
    if grid[r][c] not in ["🧀", "⚡"]:
        grid[r][c] = str(i)

for row in grid:
    print(" ".join(f"{cell:>3}" for cell in row))

print(f"\n走了 {len(path)-1} 步，最终奖励 = {reward}")
if reward == 10:
    print("✅ 老鼠吃到奶酪了！")
else:
    print("❌ 老鼠掉陷阱了……")

 19   ·   ·   ·
  ·   ⚡   ·   ·
  ·   ·   ⚡   ·
  ·   ·   ·   🧀

走了 19 步，最终奖励 = 0.0
❌ 老鼠掉陷阱了……


## 4. 实验：改参数看效果

改下面参数，重新跑上面两个 cell，看老鼠的路径怎么变：

- **EPSILON=0.0**（完全不探索）→ 老鼠容易在原地打转
- **EPSILON=0.5**（50% 随机）→ 老鼠到处乱跑，学得很慢
- **GAMMA=0.1**（只看眼前）→ 老鼠看不到奶酪的远期价值，只躲陷阱
- **LEARNING_RATE=0.9**（步子太大）→ 得分在陷阱和奶酪之间剧烈震荡

## 5. 从 Q 表到神经网络

这个网格世界里只有 16 个格子，Q 表只有 64 个数字。但 G1 呢？

G1 的状态有 29 个关节的角度 + 速度 + 姿态……不是"16 个格子"，而是**连续空间**——有无穷多种姿势。

Q 表装不下无穷多种状态。**所以用神经网络代替 Q 表**：
- 输入：一个向量（29 个关节角度、速度、身体倾斜……）
- 输出：每个关节"这样动大概能得多少分"
- 神经网络自己学习从"输入"到"输出"的映射

**Q 表是"翻字典"，神经网络是"看图识物"**——这两者的本质区别就是传统 RL 和深度 RL 的跨越。

👉 下一个 notebook：[03_dqn_cartpole.ipynb](03_dqn_cartpole.ipynb) — 用一个小神经网络控制 CartPole 平衡杆。